<span style="font-size:10pt">RDVS Numériques" du département DuMAS de l’I2M / Mars 5, 2026<br> 
v1.0  - CC BY-SA 4.0 Jean-Luc CHARLES (Jean-Luc.charles@mailo.com)</span>

<div class="alert alert-block alert-danger">
<span style="color:brown;font-family:arial;font-size:normal">
     It is important to define a <span style="font-weight:bold;">Python Virtual Environment</span> (PVE) for each Python project: a PVE makes it possible to control for each project the versions of the Python interpreter and “sensitive” modules (like tensorflow for example).<br> 
$\leadsto$ This notebook is run with the command `uv run jupyter lab` to ensure it uses the PVE of the projet.

<span style="font-family:arial;font-size:1cm;">
    Machine learning with tensorflow2/keras Python modules
</span>

# Training a simple Dense Neural Network to classify small images

## 0 - Preliminaries

### Import the Python modules

In [ ]:
# suppress tensorflow verbose warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
# Deep Learning modules:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# General modules:
import numpy as np
import matplotlib.pyplot as plt
import sys
import random
import cv2
from time import time
from pathlib import Path
from cpuinfo import get_cpu_info

# Custom modules:
from utils.tools import scan_dir, plot_images, plot_loss_accuracy, elapsed_time_since, show_conf_matrix, plot_proportion_bar

In [ ]:
print(f"Python    : {sys.version.split()[0]}")
print(f"tensorflow: {tf.__version__} with keras {keras.__version__}")
print(f"numpy     : {np.__version__}")
print(f"OpenCV    : {cv2.__version__}")

In [ ]:
# allows to visualize the graphs directly in the cell of the N.B.
%matplotlib inline

# SEED will be used to fix the _seed_ of the random generators to have continuations
# of repeatable random numbers
SEED = 1234

tf.get_logger().setLevel('ERROR')

### Create the `models` directory

In [ ]:
if Path.cwd().name != 'Notebooks':
    print("You should save & close this notebook and type 'uv run jupyter lab'")
    print("from the repository root directory ('Micrograph_Classification')")
else:
    model_path = Path("./models")
    model_path.mkdir(exist_ok=True)
    print(f"Directory `{model_path.name} successfuly created.")

## 1 - Prepare the MNIST dataset (images and labels)

#### Loading of the data

We use the keras `load_data` function to load the data from the MNIST 
(see [tf.keras.datasets.mnist.load_data](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data)):<br>
- `train_img`, `train_lab` are the training images and labels,
- `valid_img`, `valid_lab` are the validation images and labels.

In [ ]:
(train_img, train_lab), (valid_img, valid_lab) = keras.datasets.mnist.load_data()

Let's check the `shape` and `dtype` attributes of the numpy arrays:

In [ ]:
print(f"train_img.shape:  {train_img.shape}, dtype: {train_img.dtype}")
print(f"valid_img.shape:  {valid_img.shape}, dtype: {valid_img.dtype}")
print(f"train_lab.shape: {train_lab.shape}, dtype: {train_lab.dtype}")
print(f"lab_vaild.shape: {valid_lab.shape}, dtype: {valid_lab.dtype}")

### Visualization of images and labels:

The local module `utils.tools` defines the `plot_images` function which can be used to display a grid of MNIST images.<br>
Let's plot a grid of images 3 x 10 beginning with the 600th image:

In [ ]:
plot_images(train_img, 3, 10, 599, label_array=train_lab)

### Create the 3 datasets: train, validation & test

To follow the _state of the art_, we will split the dataset into train, validation & test datasets.<br>
A simple way to do this is to keep the train dataset and to split the current validation dataset in two equal subsets:
- a new smaller validation subset,
- a new test subset.

In [ ]:
# note on train_test_split : Stratified train/test split is not implemented for shuffle=False 
# We give the seed value with the 'random_state' parameter to ensure a reproducible splitting.

valid_img, test_img, valid_lab, test_lab = train_test_split(valid_img, valid_lab,
                                                            stratify=valid_lab,
                                                            test_size=0.5,
                                                            shuffle=True,
                                                            random_state=SEED)

Let's check the sizes of the 3 datasets:

In [ ]:
print(f'train:  {train_img.shape}')
print(f'valid:  {valid_img.shape}')
print(f'test :  {test_img.shape}')

We can verify that the proportion of digits remains homogenous in all the datasets:

In [ ]:
prop = {}
prop['valid'] = [ (valid_lab == i).sum() for i in range(10)]
prop['test']  = [ (test_lab  == i).sum() for i in range(10)]
plot_proportion_bar(prop, range(10));

### Define important parameters

To avoid hard-coding the number of training, valid and test images as well as the size of the images, these parameters are recovered from the data set:
- with the shape attribute of the `train_img` and `test_im` arrays
- with the size attribute of the first training image for example


In [ ]:
NB_TRAIN_IMG = train_img.shape[0] # number of training images
NB_VALID_IMG = valid_img.shape[0] # number of validation images 
NB_TEST_IMG  = test_img.shape[0]  # number of test images

NB_PIXEL     = train_img[0].size  # number of elements (pixels) of the firts training image: 

# Display checking:
print(f"{NB_TRAIN_IMG} training images, {NB_VALID_IMG} validation images and {NB_TEST_IMG} test images")
print(f"{train_img.shape[1]}x{train_img.shape[2]}={NB_PIXEL} pixels in each image")

# number of classes:
NB_CLASS = len(set(train_lab))
print(f"{NB_CLASS} classes found in the `train_lab` ndarray")

### Check: display the shapes of the dataset arrays

In [ ]:
print(f'train:  {train_img.shape}')
print(f'valid:  {valid_img.shape}')
print(f'test :  {test_img.shape}')

## 3 - Process input data

Two treatments must be applied to the data from the MNIST database:
- on the images: transform the matrices of  28$\,\times\,$28 pixels (`uint8`integers) into **normalized** vectors $(V_i)_{i=0..783}$ of 784 real values $V_i$ with $ 0 \leqslant V_i \leqslant 1$;
- on labels: transform scalar numbers into *one-hot* vectors.

### Transform input matrices into normalized vectors

We define the arrays `x_train`, `x_valid` and `x_test` containing the matrices of the arrays `train_img`, `valid_img` and `test_img` *flattened* as normalized vectors (values between 0 and 1):

In [ ]:
x_train = train_img.reshape(NB_TRAIN_IMG, NB_PIXEL)/255
x_valid = valid_img.reshape(NB_VALID_IMG, NB_PIXEL)/255
x_test  = test_img.reshape(NB_TEST_IMG, NB_PIXEL)/255

#check:
print(f'train: {x_train.shape}, min: {x_train.min()}, max: {x_train.max()}')
print(f'valid: {x_valid.shape}, min: {x_valid.min()}, max: {x_valid.max()}')
print(f'test : {x_test.shape}, min: {x_test.min()}, max: {x_test.max()}')

### *one-hot* encoding of labels:

We use the **keras** `to_categorical` function (see [tf.keras.utils.to_categorical](https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical)) to define the `y_train` and `y_valid` arrays containing the *hot-one* encoded version of `lab_train` and `lab_valid`:

In [ ]:
from tensorflow.keras.utils import to_categorical
# 'one-hot' encoding' of labels :
y_train = to_categorical(train_lab)
y_valid = to_categorical(valid_lab)
y_test  = to_categorical(test_lab)

Let's check the first 10 values of the `lab_train` and `y_train` arrays:

In [ ]:
print(train_lab[:10])
print(y_train[:10])

## 4 - Build the Dense Neural Network (DNN)

 To get short computation times we build a simple dense network to classify the MNIST images.<br>
 Of course, this is not the "state of the art" : convolutive NN, transformers have much more impressive scores, but we just want want short training computation time.<br><br>
We buildthis naive **dense network**:
- an **input layer** of 784 values (the pixels of the MNIST 28 $\times$ 28 images flattened in the form of a vector of 784 normalized `float` numbers),
- a **hidden layer** of 784 neurons using the `relu` activation function,
- an **output layer** of 10 neurons, for the classification of the 10 digits {0,1,2...9}, using the `softmax` activation function adapted to classification problems .

<p style="text-align:center; font-style:italic; font-size:12px;">
      <img src="img/Simple-DNN.png" alt="Simple-DNN.png" style="width:900px;"><br>
     [image credit: JLC]

In [ ]:
NB_INPUT  = NB_PIXEL
NB_NEURON = NB_PIXEL

For the sake of convenience we défine a function to build the NN: 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def build_DNN(seed=None):

    if seed is not None:
        ##########################
        # Deterministic training #
        ##########################
        # 1/ set the seed of the random generators involved by tensorflow:
        tf.keras.utils.set_random_seed(seed)
        # 2/ make the tf ops determinisctic 
        # [see https://blog.tensorflow.org/2022/05/whats-new-in-tensorflow-29.html]
        tf.config.experimental.enable_op_determinism() 

    model = Sequential()
    model.add(Input(shape=(NB_INPUT,), name='input'))             # INPUT layer
    model.add(Dense(NB_NEURON, activation='relu', name='c1'))     # First hidden layer
    model.add(Dense(NB_CLASS, activation='softmax', name='c2'))   # OUTPUT layer
    model.compile(loss='categorical_crossentropy', optimizer='adam',  metrics=['accuracy'])
    return model

Lets's look at the number of _parameters_ (the _weights_) of the model:

In [ ]:
model = build_DNN()
model.summary()

## 5 - Elementary tests of the reproducibility of the DNN training...

Perfect reproducibility of a neural network training may be difficult to achieve with tensorflow...<br>
To highlight the question of reproducibility we run a short experiment: <br>
$\leadsto$ run a loop where the model is built without setting the seed, trained and evaluated once (_epochs=1_) at each iteration.

#### A/ Build the model at each lop lap without SEED: training not reproducible

In [ ]:
print(get_cpu_info()['brand_raw'])
for _ in range(5):
    model = build_DNN()  # Build a new model without setting seed
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

In the previous cell we focus on the metrics `val_accuracy`and `val_loss` to evaluate the training after one epoch: we see that if we repeat 5 times the first epoch of the training, the `val_accuracy`and `val_loss` values differ at each iteration $\leadsto$ the training is not reproducible.

#### B/ Reload a initial model weights without setting SEED: training not reproducible

We save the weights of a model built without setting the seed:

In [ ]:
model = build_DNN()
model.save_weights('models/DNN_noseed.weights.h5')

In [ ]:
print(get_cpu_info()['brand_raw'])
model = build_DNN()
for _ in range(5):
    model.load_weights('models/DNN_noseed.weights.h5') # reload the inital model weights
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

The `val_loss` is still non-reproducible.

#### C/ Build the model with SEED at each lop lap: training perfectly reproducible

In [ ]:
print(get_cpu_info()['brand_raw'])
for _ in range(5):
    model = build_DNN(seed=1234)  # Build a new model with seed set
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

If we set the seed when creating the model at each iteration the `val_accuracy`and `val_loss` are perfectly reproducible &#128526;.

#### D/ Build the model with SEED  and load the model's structure & initial at each loop lap: training perfectly reproducible

In [ ]:
model = build_DNN(seed=1234)
model.save('models/DNN_seed1234.keras')

In [ ]:
print(get_cpu_info()['brand_raw'])
for _ in range(5):
    model = tf.keras.models.load_model('models/DNN_seed1234.keras') # reload the model structure & weights 
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

If we load the structure and the weights of the model at each iteration the `val_accuracy`and `val_loss` are perfectly reproducible  &#128526;.

<span style="color:brown">We can ensure the reproducibility of the first epoch of the training:<br>
$\leadsto$ by setting the seed at the model creation (and everywhere else where required)<br>
$\leadsto$ by reloading the entire network structure and weights with the __tf.keras.models.load_model__ function<br><br>
</spawn>

## 6 - Adressing the reproducibility of the DNN training...

### A - Run several trainings with a new model (no SEED) at each iteration $\leadsto$ not reproducible

To avoid long computation time, you can skip running the next cell and just look at the result obtained on a core-i7 standard laptop:

<p style="text-align:center; font-style:italic; font-size:12px;">
      <img src="img/range5_noSEED_epochs10_bs32.png" alt="range5_noSEED_epochs10_bs32.png" style="width:1024px;"><br>
     [image credit: JLC]

In [ ]:
H, t0 = [], time()
print("training ", end="")

for i in range(5):
    print(f" #{i+1}", end="")
    
    model = build_DNN()                # build a new network, no SEED given
    
    hist = model.fit(x_train, y_train, # images, labels
                     epochs=10,        # the total number of successive trainings
                     batch_size=32,    # split the whole dadaset in batches
                     validation_data=(x_valid, y_valid), 
                     verbose=0)
    H.append(hist)  

print(f' Total Train {elapsed_time_since(t0)}')   
plot_loss_accuracy(H);

The `val_loss` and `val_accuracy` differ at each of the training.<br>
It can be a problem if we train the model with a _callback_ like __early stoppping__ : the training will stop at a different epoch if we run the training many times.

### B - New model with SEED at each loop lap: $\leadsto$ reproducible

To avoid long computation time, you can skip running the next cell and just look at the result obtained on a core-i7 standard laptop:

<p style="text-align:center; font-style:italic; font-size:12px;">
      <img src="img/range5_SEED1234_epochs10_bs32.png" alt="range5_SEED1234_epochs10_bs32.png" style="width:1024px;"><br>
     [image credit: JLC]

In [ ]:
H, t0 = [], time()
print("training ", end="")

for i in range(5):
    print(f" #{i+1}", end="")
    
    # build a new network/
    model = build_DNN(1234)            # build a new network with a fixed SEED
    
    hist = model.fit(x_train, y_train, # images, labels
                     epochs=10,        # the total number of successive trainings
                     batch_size=32,    # split of the whole dadaset in batches
                     validation_data=(x_valid, y_valid), 
                     verbose=0)
    H.append(hist)  

print(f' Total Train {elapsed_time_since(t0)}')
plot_loss_accuracy(H);

$\leadsto$ the repoducibility is perfect &#128526;

### C - Reloading the model structure & weights with SEED at each loop lap $\leadsto$ reproducible

To avoid long computation time, you can skip running the next cell and just look at the result obtained on a core-i7 standard laptop:

<p style="text-align:center; font-style:italic; font-size:12px;">
      <img src="img/range5_reload_SEED1234_epochs10_bs32.png" alt="range5_reload_SEED1234_epochs10_bs32.png" style="width:1024px;"><br>
     [image credit: JLC]

In [ ]:
model = build_DNN(seed=1234)
model.save('models/DNN_seed1234.keras')

In [ ]:
H, t0 = [], time()
print("training ", end="")

for i in range(5):
    print(f" #{i+1}", end="")
                
    # reload the structure of the NN and its initial state 
    model = tf.keras.models.load_model('models/DNN_seed1234.keras') 
    
    # set the seed at each turn of the loop:
    tf.random.set_seed(1234)
    
    # train the network
    hist = model.fit(x_train, y_train, 
                     epochs=10, 
                     batch_size=32,
                     validation_data=(x_valid, y_valid), 
                     verbose=0)
    H.append(hist)  

print(f' Total Train {elapsed_time_since(t0)}')   
plot_loss_accuracy(H);

## 7 - Train the network

### Automatically stop training before over-fit
Keras offers tools to automatically stop learning by monitoring for example the growth of `val_accuracy` or the decrease of `val_loss` from one epoch to another (see the _EarlyStopping_ callback).

We can thus define a list of callback functions that we pass as an argument to the `fit` method with the agument named _callbacks_:

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# define the list of 'callback' fucntions:
callbacks_list = [
    EarlyStopping(monitor='val_loss',  # The parameter to monitor
                  patience=2,           # accept that 'val_accuracy' decrease 2 times
                  restore_best_weights=True,
                  verbose=1)
]

# load the network structure & initial weights:
model = tf.keras.models.load_model('models/DNN_seed1234.keras')

# Deterministic tensorflow training: 
# see https://blog.tensorflow.org/2022/05/whats-new-in-tensorflow-29.html
tf.keras.utils.set_random_seed(SEED)  # sets seeds for base-python, numpy and tf
tf.config.experimental.enable_op_determinism() 

hist = model.fit(x_train, y_train,
                 validation_data=(x_valid, y_valid), 
                 epochs=25,     # the total number of successive trainings
                 batch_size=32, # fragmentation of the whole dada set in batches
                 callbacks = callbacks_list)
plot_loss_accuracy(hist);

### Save the trained model

In [ ]:
model.save('models/DNN_seed1234_trained.keras')

## 8 - Evaluate the trained network <a name="6"></a>

### Show the confusion matrix

In [ ]:
model = tf.keras.models.load_model('models/DNN_seed1234_trained.keras')

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)

# Predicting labels for test images
predict_labels = np.argmax(model.predict(x_test), axis=-1)

# Display classification report
print("Classification Report:\n", classification_report(np.argmax(y_test, axis=-1), predict_labels))

In [ ]:
show_conf_matrix(test_lab, predict_labels, range(NB_CLASS), figsize=(7,6));